# 2026 Spurs series preview — Wembanyama minutes

Victor Wembanyama: count of **2025-26** games (Regular Season + Playoffs) with **at least 38 minutes** played.

Uses the same `PlayerGameLogs` + `GAME_ID` filtering approach as `in-game/bones-without-ant.ipynb`.


In [11]:
import pandas as pd
from IPython.display import display
from nba_api.stats.static import players as nba_players
from nba_api.stats.endpoints import PlayerGameLogs

SEASON = "2025-26"

matches = nba_players.find_players_by_full_name("Victor Wembanyama")
if len(matches) != 1:
    raise ValueError(
        f"Expected exactly one Victor Wembanyama; got {len(matches)}. "
        "Set PLAYER_ID = 1641705 manually if needed."
    )
player_id = matches[0]["id"]

frames = []
for season_type in ("Regular Season", "Playoffs"):
    df = PlayerGameLogs(
        player_id_nullable=player_id,
        season_nullable=SEASON,
        season_type_nullable=season_type,
        timeout=90,
    ).get_data_frames()[0]
    if len(df) > 0:
        frames.append(df)

wemby_logs = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()


In [15]:

MIN_THRESHOLD = 40
if len(wemby_logs) > 0:
    wemby_logs = wemby_logs[
        wemby_logs["GAME_ID"].astype(str).str[2].isin(["2", "4"])
    ].copy()
    wemby_logs = wemby_logs.drop_duplicates(subset=["GAME_ID"], keep="first")

games_38_plus = wemby_logs[wemby_logs["MIN"] >= MIN_THRESHOLD].sort_values(
    "GAME_DATE", ascending=False
)

print(f"Victor Wembanyama (PLAYER_ID={player_id})")
print(f"Season {SEASON} — Regular Season + Playoffs (GAME_ID types 2 & 4 only)")
print("=" * 70)
print(f"Total games in sample: {len(wemby_logs)}")
print(f"Games with MIN >= {MIN_THRESHOLD}: {len(games_38_plus)}")
print()

if len(games_38_plus) > 0:
    show_cols = [c for c in ("GAME_DATE", "MATCHUP", "WL", "MIN", "PTS", "REB", "AST", "TOV", "STL", "BLK") if c in games_38_plus.columns]
    display(games_38_plus[show_cols])


Victor Wembanyama (PLAYER_ID=1641705)
Season 2025-26 — Regular Season + Playoffs (GAME_ID types 2 & 4 only)
Total games in sample: 68
Games with MIN >= 40: 0



In [7]:
# Timberwolves vs Spurs (2025-26 regular season) — points leaders (aggregated)

import numpy as np
from IPython.display import display
from nba_api.stats.endpoints import PlayerGameLogs

TEAM_ID_MIN = "1610612750"  # Minnesota Timberwolves
OPP_SAS = "1610612759"  # San Antonio Spurs
SEASON = "2025-26"

wolves_vs_sas = PlayerGameLogs(
    team_id_nullable=TEAM_ID_MIN,
    season_nullable=SEASON,
    season_type_nullable="Regular Season",
    opp_team_id_nullable=OPP_SAS,
    timeout=90,
).get_data_frames()[0]

if len(wolves_vs_sas) > 0:
    wolves_vs_sas = wolves_vs_sas[
        wolves_vs_sas["GAME_ID"].astype(str).str[2] == "2"
    ].copy()

if len(wolves_vs_sas) == 0:
    print("No game logs returned for MIN @/vs SAS in 2025-26 Regular Season.")
else:
    g = wolves_vs_sas.groupby(["PLAYER_ID", "PLAYER_NAME"], as_index=False)
    agg = g.agg(
        GP=("GAME_ID", "count"),
        PTS=("PTS", "sum"),
        FGM=("FGM", "sum"),
        FGA=("FGA", "sum"),
        _3PM=("FG3M", "sum"),
        _3PA=("FG3A", "sum"),
    )
    agg["2PM"] = agg["FGM"] - agg["_3PM"]
    agg["2PA"] = agg["FGA"] - agg["_3PA"]
    agg["FG%"] = np.where(agg["FGA"] > 0, agg["FGM"] / agg["FGA"], np.nan)
    agg["2P%"] = np.where(agg["2PA"] > 0, agg["2PM"] / agg["2PA"], np.nan)
    agg["3P%"] = np.where(agg["_3PA"] > 0, agg["_3PM"] / agg["_3PA"], np.nan)

    out = agg.rename(columns={"_3PM": "3PM", "_3PA": "3PA"})
    out = out[
        [
            "PLAYER_NAME",
            "GP",
            "PTS",
            "FGM",
            "FGA",
            "FG%",
            "2PM",
            "2PA",
            "2P%",
            "3PM",
            "3PA",
            "3P%",
        ]
    ].sort_values("PTS", ascending=False)
    out = out.reset_index(drop=True)

    for c in ("FG%", "2P%", "3P%"):
        out[c] = out[c].round(3)
    for c in ("GP", "PTS", "FGM", "FGA", "2PM", "2PA", "3PM", "3PA"):
        out[c] = out[c].astype(int)

    print("Timberwolves vs San Antonio Spurs — 2025-26 Regular Season (totals across matchup games)")
    print("=" * 80)
    display(out)


Timberwolves vs San Antonio Spurs — 2025-26 Regular Season (totals across matchup games)


,PLAYER_NAME,GP,PTS,FGM,FGA,FG%,2PM,2PA,2P%,3PM,3PA,3P%
0,Anthony Edwards,3,110,42,72,0.583,28,45,0.622,14,27,0.519
1,Julius Randle,3,54,17,39,0.436,15,29,0.517,2,10,0.200
2,Donte DiVincenzo,3,48,18,44,0.409,6,13,0.462,12,31,0.387
3,Jaden McDaniels,3,48,18,36,0.500,16,27,0.593,2,9,0.222
4,Naz Reid,3,32,13,30,0.433,7,16,0.438,6,14,0.429
5,Bones Hyland,3,17,6,11,0.545,3,3,1.000,3,8,0.375
6,Rudy Gobert,2,10,5,10,0.500,5,10,0.500,0,0,NaN
7,Jaylen Clark,3,10,3,5,0.600,1,2,0.500,2,3,0.667
8,Joan Beringer,1,10,5,6,0.833,5,6,0.833,0,0,NaN
9,Mike Conley,3,9,3,9,0.333,1,4,0.250,2,5,0.400


In [8]:
# Anthony Edwards & Donte DiVincenzo vs Spurs — vs the rest of the Wolves (2025-26 RS)

import numpy as np
import pandas as pd
from IPython.display import display
from nba_api.stats.endpoints import PlayerGameLogs

TEAM_ID_MIN = "1610612750"
OPP_SAS = "1610612759"
SEASON = "2025-26"

wolves_vs_sas = PlayerGameLogs(
    team_id_nullable=TEAM_ID_MIN,
    season_nullable=SEASON,
    season_type_nullable="Regular Season",
    opp_team_id_nullable=OPP_SAS,
    timeout=90,
).get_data_frames()[0]

if len(wolves_vs_sas) > 0:
    wolves_vs_sas = wolves_vs_sas[
        wolves_vs_sas["GAME_ID"].astype(str).str[2] == "2"
    ].copy()

if len(wolves_vs_sas) == 0:
    print("No MIN vs SAS regular-season logs.")
else:
    n_games = wolves_vs_sas["GAME_ID"].nunique()

    def row_from_slice(df):
        if len(df) == 0:
            return None
        pts = int(df["PTS"].sum())
        fgm = int(df["FGM"].sum())
        fga = int(df["FGA"].sum())
        t3m = int(df["FG3M"].sum())
        t3a = int(df["FG3A"].sum())
        pm2 = fgm - t3m
        pa2 = fga - t3a
        return {
            "GP": int(df["GAME_ID"].nunique()),
            "PTS": pts,
            "FGM": fgm,
            "FGA": fga,
            "FG%": round(fgm / fga, 3) if fga else np.nan,
            "2PM": pm2,
            "2PA": pa2,
            "2P%": round(pm2 / pa2, 3) if pa2 else np.nan,
            "3PM": t3m,
            "3PA": t3a,
            "3P%": round(t3m / t3a, 3) if t3a else np.nan,
        }

    ed = wolves_vs_sas[wolves_vs_sas["PLAYER_NAME"] == "Anthony Edwards"]
    dd = wolves_vs_sas[wolves_vs_sas["PLAYER_NAME"] == "Donte DiVincenzo"]
    rest = wolves_vs_sas[
        ~wolves_vs_sas["PLAYER_NAME"].isin(["Anthony Edwards", "Donte DiVincenzo"])
    ]

    team_pts = int(wolves_vs_sas["PTS"].sum())
    team_fgm = int(wolves_vs_sas["FGM"].sum())

    rows = []
    for label, slice_df in (
        ("Anthony Edwards", ed),
        ("Donte DiVincenzo", dd),
        ("Rest of team", rest),
    ):
        r = row_from_slice(slice_df)
        if r is None:
            continue
        r["segment"] = label
        r["PTS_share"] = round(r["PTS"] / team_pts, 3) if team_pts else np.nan
        r["FGM_share"] = round(r["FGM"] / team_fgm, 3) if team_fgm else np.nan
        rows.append(r)

    cmp_df = pd.DataFrame(rows)
    cmp_df = cmp_df[
        [
            "segment",
            "GP",
            "PTS",
            "PTS_share",
            "FGM",
            "FGM_share",
            "FGA",
            "FG%",
            "2PM",
            "2PA",
            "2P%",
            "3PM",
            "3PA",
            "3P%",
        ]
    ]

    duo_pts = int(ed["PTS"].sum()) + int(dd["PTS"].sum())
    duo_fgm = int(ed["FGM"].sum()) + int(dd["FGM"].sum())

    print(
        f"Minnesota vs San Antonio — {SEASON} regular season "
        f"({n_games} games). Team PTS in sample: {team_pts}"
    )
    print(
        f"Edwards + DiVincenzo: {duo_pts} PTS ({duo_pts / team_pts:.1%} of team) | "
        f"Rest of roster: {team_pts - duo_pts} PTS ({(team_pts - duo_pts) / team_pts:.1%} of team)"
    )
    print("=" * 80)
    display(cmp_df)


Minnesota vs San Antonio — 2025-26 regular season (3 games). Team PTS in sample: 352
Edwards + DiVincenzo: 158 PTS (44.9% of team) | Rest of roster: 194 PTS (55.1% of team)


,segment,GP,PTS,PTS_share,FGM,FGM_share,FGA,FG%,2PM,2PA,2P%,3PM,3PA,3P%
0,Anthony Edwards,3,110,0.312,42,0.323,72,0.583,28,45,0.622,14,27,0.519
1,Donte DiVincenzo,3,48,0.136,18,0.138,44,0.409,6,13,0.462,12,31,0.387
2,Rest of team,3,194,0.551,70,0.538,151,0.464,53,99,0.535,17,52,0.327


In [9]:
# Spurs vs Timberwolves (2025-26 regular season) — per-player stats (aggregated)

import numpy as np
import pandas as pd
from IPython.display import display
from nba_api.stats.endpoints import PlayerGameLogs

TEAM_ID_SAS = "1610612759"  # San Antonio Spurs
OPP_MIN = "1610612750"  # Minnesota Timberwolves
SEASON = "2025-26"

sas_vs_min = PlayerGameLogs(
    team_id_nullable=TEAM_ID_SAS,
    season_nullable=SEASON,
    season_type_nullable="Regular Season",
    opp_team_id_nullable=OPP_MIN,
    timeout=90,
).get_data_frames()[0]

if len(sas_vs_min) > 0:
    sas_vs_min = sas_vs_min[
        sas_vs_min["GAME_ID"].astype(str).str[2] == "2"
    ].copy()

if len(sas_vs_min) == 0:
    print("No game logs returned for SAS @/vs MIN in 2025-26 Regular Season.")
else:
    g = sas_vs_min.groupby(["PLAYER_ID", "PLAYER_NAME"], as_index=False)
    agg = g.agg(
        GP=("GAME_ID", "count"),
        MIN=("MIN", "mean"),
        PTS=("PTS", "sum"),
        FGM=("FGM", "sum"),
        FGA=("FGA", "sum"),
        _3pm=("FG3M", "sum"),
        _3pa=("FG3A", "sum"),
        REB=("REB", "sum"),
        AST=("AST", "sum"),
        TOV=("TOV", "sum"),
        STL=("STL", "sum"),
        BLK=("BLK", "sum"),
    )
    agg["2PM"] = agg["FGM"] - agg["_3pm"]
    agg["2PA"] = agg["FGA"] - agg["_3pa"]
    agg["FG%"] = np.where(agg["FGA"] > 0, agg["FGM"] / agg["FGA"], np.nan)
    agg["2P%"] = np.where(agg["2PA"] > 0, agg["2PM"] / agg["2PA"], np.nan)
    agg["3P%"] = np.where(agg["_3pa"] > 0, agg["_3pm"] / agg["_3pa"], np.nan)

    out = agg.rename(columns={"_3pm": "3PM", "_3pa": "3PA"})
    out = out[
        [
            "PLAYER_NAME",
            "GP",
            "MIN",
            "PTS",
            "FGM",
            "FGA",
            "FG%",
            "2PM",
            "2PA",
            "2P%",
            "3PM",
            "3PA",
            "3P%",
            "REB",
            "AST",
            "TOV",
            "STL",
            "BLK",
        ]
    ].sort_values("PTS", ascending=False)
    out = out.reset_index(drop=True)

    for c in ("FG%", "2P%", "3P%"):
        out[c] = out[c].round(3)
    out["MIN"] = out["MIN"].round(1)
    for c in (
        "GP",
        "PTS",
        "FGM",
        "FGA",
        "2PM",
        "2PA",
        "3PM",
        "3PA",
        "REB",
        "AST",
        "TOV",
        "STL",
        "BLK",
    ):
        out[c] = out[c].astype(int)

    print(
        "Spurs vs Minnesota Timberwolves — 2025-26 Regular Season (totals across matchup games; "
        "MIN = average minutes per game)"
    )
    print("=" * 80)
    display(out)


Spurs vs Minnesota Timberwolves — 2025-26 Regular Season (totals across matchup games; MIN = average minutes per game)


,PLAYER_NAME,GP,MIN,PTS,FGM,FGA,FG%,2PM,2PA,2P%,3PM,3PA,3P%,REB,AST,TOV,STL,BLK
0,Victor Wembanyama,2,28.8,68,20,41,0.488,13,23,0.565,7,18,0.389,16,4,3,3,2
1,De'Aaron Fox,3,34.1,62,26,51,0.510,22,36,0.611,4,15,0.267,13,20,7,2,1
2,Keldon Johnson,3,27.9,57,21,35,0.600,13,22,0.591,8,13,0.615,11,4,3,3,0
3,Harrison Barnes,3,29.9,26,10,27,0.370,4,11,0.364,6,16,0.375,14,4,3,1,0
4,Julian Champagnie,3,31.1,26,8,22,0.364,3,4,0.750,5,18,0.278,19,4,5,2,0
5,Dylan Harper,3,17.4,26,12,26,0.462,11,18,0.611,1,8,0.125,4,4,1,2,0
6,Devin Vassell,1,31.7,22,10,20,0.500,9,11,0.818,1,9,0.111,1,3,1,2,0
7,Luke Kornet,3,22.0,20,8,13,0.615,8,13,0.615,0,0,NaN,22,5,2,4,4
8,Stephon Castle,2,31.3,19,4,19,0.211,2,14,0.143,2,5,0.400,12,14,9,3,0
9,Kelly Olynyk,2,8.8,7,3,4,0.750,2,2,1.000,1,2,0.500,6,1,1,0,0


In [2]:
# Spurs record when Wembanyama has elevated 3-point volume (2025-26)

import pandas as pd
from IPython.display import display
from nba_api.stats.static import players as nba_players
from nba_api.stats.endpoints import PlayerGameLogs

SEASON = "2025-26"
THRESHOLDS = (5, 7, 10)

matches = nba_players.find_players_by_full_name("Victor Wembanyama")
if len(matches) != 1:
    raise ValueError(
        f"Expected exactly one Victor Wembanyama; got {len(matches)}. "
        "Set PLAYER_ID = 1641705 manually if needed."
    )
player_id = matches[0]["id"]

frames = []
for season_type in ("Regular Season", "Playoffs"):
    df = PlayerGameLogs(
        player_id_nullable=player_id,
        season_nullable=SEASON,
        season_type_nullable=season_type,
        timeout=90,
    ).get_data_frames()[0]
    if len(df) > 0:
        frames.append(df)

wemby_logs = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

if len(wemby_logs) > 0:
    wemby_logs = wemby_logs[
        wemby_logs["GAME_ID"].astype(str).str[2].isin(["2", "4"])
    ].copy()

print("San Antonio Spurs — record when Victor Wembanyama attempts ≥N three-pointers (FG3A)")
print(
    f"Season {SEASON}, Regular Season + Playoffs | sample: all games Wembanyama played (n={len(wemby_logs)})"
)
print("=" * 72)

if len(wemby_logs) == 0:
    print("No game logs.")
else:
    shoot_rows = []
    for thr in THRESHOLDS:
        sub = wemby_logs[wemby_logs["FG3A"] >= thr]
        wins = (sub["WL"] == "W").sum()
        losses = (sub["WL"] == "L").sum()
        gp = wins + losses
        pct = wins / gp if gp else float("nan")
        print(
            f"  FG3A >= {thr:2d}:  {gp:3d} gp  |  {wins}-{losses}  |  {pct:.3f} ({pct * 100:.1f}%)"
        )
        fg3m = int(sub["FG3M"].sum())
        fg3a = int(sub["FG3A"].sum())
        tpct = (fg3m / fg3a) if fg3a else float("nan")
        shoot_rows.append(
            {
                "min_3PA": thr,
                "GP": gp,
                "FG3M": fg3m,
                "FG3A": fg3a,
                "3P%": round(tpct, 3) if fg3a else float("nan"),
            }
        )

    shoot_df = (
        pd.DataFrame(shoot_rows)
        .sort_values("min_3PA", ascending=False)
        .reset_index(drop=True)
    )
    print()
    print("Wembanyama aggregate 3P in those game sets (highest 3PA floor first):")
    display(shoot_df)


San Antonio Spurs — record when Victor Wembanyama attempts ≥N three-pointers (FG3A)
Season 2025-26, Regular Season + Playoffs | sample: all games Wembanyama played (n=69)
  FG3A >=  5:   39 gp  |  28-11  |  0.718 (71.8%)
  FG3A >=  7:   23 gp  |  16-7  |  0.696 (69.6%)
  FG3A >= 10:    5 gp  |  4-1  |  0.800 (80.0%)

Wembanyama aggregate 3P in those game sets (highest 3PA floor first):


,min_3PA,GP,FG3M,FG3A,3P%
0,10,5,26,57,0.456
1,7,23,72,201,0.358
2,5,39,110,292,0.377


In [1]:
# Opponent record @ Spurs when the opponent attempts ≥N team 3-pointers (2025-26)

import time
import pandas as pd
from IPython.display import display
from nba_api.stats.endpoints import BoxScoreTraditionalV3, TeamGameLogs

SAS_ID = 1610612759
SEASON = "2025-26"
THRESHOLDS = (30, 33, 35, 37, 40)
SLEEP_S = 0.5  # be gentle to stats.nba.com (~0.5s between box score calls)

frames = []
for season_type in ("Regular Season", "Playoffs"):
    df = TeamGameLogs(
        team_id_nullable=str(SAS_ID),
        season_nullable=SEASON,
        season_type_nullable=season_type,
        timeout=90,
    ).get_data_frames()[0]
    if len(df) > 0:
        frames.append(df)

spurs_logs = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
if len(spurs_logs) > 0:
    spurs_logs = spurs_logs[
        spurs_logs["GAME_ID"].astype(str).str[2].isin(["2", "4"])
    ].copy()
    spurs_logs = spurs_logs.drop_duplicates(subset=["GAME_ID"], keep="first")

per_game = []
failed = 0
for idx, r in spurs_logs.iterrows():
    gid = str(r["GAME_ID"])
    spurs_wl = r.get("WL")
    if spurs_wl not in ("W", "L"):
        failed += 1
        continue
    try:
        dfs = BoxScoreTraditionalV3(game_id=gid).get_data_frames()
        if len(dfs) < 3 or dfs[2] is None or len(dfs[2]) < 2:
            failed += 1
            continue
        team_totals = dfs[2]
        opp_rows = team_totals[team_totals["teamId"].astype(int) != SAS_ID]
        if len(opp_rows) != 1:
            failed += 1
            continue
        opp = opp_rows.iloc[0]
        tpa = int(opp["threePointersAttempted"])
        opp_win = 1 if spurs_wl == "L" else 0
        opp_loss = 1 if spurs_wl == "W" else 0
        per_game.append(
            {"GAME_ID": gid, "opp_3PA": tpa, "opp_win": opp_win, "opp_loss": opp_loss}
        )
    except Exception:
        failed += 1
        continue
    time.sleep(SLEEP_S)

games_df = pd.DataFrame(per_game)

print(
    f"Opponent team 3PA from BoxScoreTraditionalV3 team totals | "
    f"Spurs games in sample: {len(spurs_logs)} | "
    f"Box scores parsed: {len(games_df)} | "
    f"skipped/failed: {failed}"
)
print("Opponent W/L is from the opponent's perspective (Spurs loss = opponent win).")
print("=" * 72)

if len(games_df) == 0:
    print("No games to summarize.")
else:
    summary_rows = []
    for thr in sorted(THRESHOLDS, reverse=True):
        sub = games_df[games_df["opp_3PA"] >= thr]
        w = int(sub["opp_win"].sum())
        l = int(sub["opp_loss"].sum())
        gp = w + l
        wpct = (w / gp) if gp else float("nan")
        summary_rows.append(
            {
                "min_opp_3PA": thr,
                "GP": gp,
                "Opp_W": w,
                "Opp_L": l,
                "Opp_W%": round(wpct, 3) if gp else float("nan"),
            }
        )
    summary_df = pd.DataFrame(summary_rows).reset_index(drop=True)
    display(summary_df)


Opponent team 3PA from BoxScoreTraditionalV3 team totals | Spurs games in sample: 89 | Box scores parsed: 89 | skipped/failed: 0
Opponent W/L is from the opponent's perspective (Spurs loss = opponent win).


,min_opp_3PA,GP,Opp_W,Opp_L,Opp_W%
0,40,26,7,19,0.269
1,37,45,12,33,0.267
2,35,54,14,40,0.259
3,33,60,16,44,0.267
4,30,74,19,55,0.257
